This is an example of obtain 1) DOC injection fluxes and 2) [DOC]-time to drive ATS 2D transect simualations

- Input
    - 1
    - 2
- Output
    - 1
    - 2

**Version History**

**2026/2/3**
- finally merged into the repo on NERSC, as previously it's a two step processing.

**2025/10/17**
- add concentration unit convertion from per bulk volume to per water volumn

**2025/10/14**
- add `config.json`

**2025/10/8**
- add hillslope average plot over time
- add NH4+, NO3- extraction

**2025/8/14**
- output DOC fluxes h5 files
    - without fdom and soil moisture scaling
    - fdom and soil moisture scaling are handled by final h5 generation for 2D hillslope model
- output soil moisture h5 files
    - used by possible scaling
- output LIT and SOM pools concententration vertically resolved
    - for potential estimation of DOC concentration

In [ ]:
%load_ext autoreload
%autoreload 2

# Parameters and data sources

In [ ]:
# Parameters cell -- schema-v2 date-based configuration
import sys
sys.path.insert(0, '..')
from config_utils import load_config, phase_dates, phase_label, phase_names, phase_period, noleap_day_of_year, phase_forcing_dir, full_timeline_forcing_dir

config = load_config('../config.json')
case = config['case']
watershed_name = case['watershed_name']
hucs = [case['hucs']]
site_name = case['site_name']
meshsize_nx = case['meshsize_nx']

spinup_dates = phase_dates(config, 'spinup')
prefire_dates = phase_dates(config, 'prefire_transient')
postfire_dates = phase_dates(config, 'postfire_transient') if 'postfire_transient' in config else []
spinup_label = phase_label(config, 'spinup')
prefire_label = phase_label(config, 'prefire_transient')
postfire_label = phase_label(config, 'postfire_transient') if postfire_dates else None
forcing_spinup_dir = phase_forcing_dir(config, 'spinup', '../../data-processed')
forcing_prefire_dir = phase_forcing_dir(config, 'prefire_transient', '../../data-processed')
forcing_postfire_dir = phase_forcing_dir(config, 'postfire_transient', '../../data-processed') if postfire_dates else None
forcing_full_dir = full_timeline_forcing_dir(config, '../../data-processed')
for _d in (forcing_spinup_dir, forcing_prefire_dir, forcing_postfire_dir, forcing_full_dir):
    if _d is not None: _d.mkdir(parents=True, exist_ok=True)

start_year_spinup = spinup_dates[0].year
end_year_spinup = spinup_dates[-1].year
nyears_steadystate_spinup = config['spinup']['steady_state_years']
nyears_cyclic_spinup = config['spinup']['cyclic_years']
start_year_transient = prefire_dates[0].year
end_year_transient = prefire_dates[-1].year
# ELM realization roles: prefire uses prefire_transient.elm_run; the one ignition-day
# sample uses postfire_transient.ignition_day_elm_run when configured; subsequent
# postfire samples use postfire_transient.elm_run.
run = config['prefire_transient']['elm_run']


In [ ]:
from scipy.io import loadmat

# Load soil_thickness_median from the mat file generated in 1a-main_workflow_Naches.ats1.5.ipynb
soil_thickness_median_file = f'../../data-processed/{site_name}/soilmedianthickness_{site_name}.mat'
loaded_data = loadmat(soil_thickness_median_file)
soil_thickness_median = loaded_data['soil_thickness_median'].item()

print(f"Loaded soil_thickness_median: {soil_thickness_median} m")

rho_m = 55000. # moles/m^3, water molar density
outputs={}

In [ ]:
import os, sys
from pathlib import Path
import glob
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import h5py as h5
from tqdm import tqdm, trange
from scipy.io import loadmat
import math


In [ ]:
import h5py
from pyproj import Transformer
from scipy.spatial import cKDTree

In [ ]:
# from makegif import make_gif
import cartopy
from herbie import Herbie #[Yi] Herbie is used by Zhi to process HRRR data?
from herbie.toolbox import EasyMap, pc
naches_wbd = np.loadtxt('./WBD_xyz/naches_wbd.xyz')
oakcreek_wbd = np.loadtxt('./WBD_xyz/oakcreek_wbd.xyz')

generate_animation = False

In [ ]:
import shapely
from shapely.geometry import Point,Polygon,mapping
import geopandas as gpd

import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.ui
import watershed_workflow.colors
import watershed_workflow.condition
import watershed_workflow.mesh
import watershed_workflow.split_hucs
import watershed_workflow.soil_properties
import watershed_workflow.daymet
import watershed_workflow.utils
import watershed_workflow.regions

# Prepare watershed shape and hillslope shape

In [ ]:
# load hillslope geometry from mat file generated in "1-full_workflow_OakCreek.ipynb"
meshsize_nx=100

m2_mat_filename =  f'../../data-processed/{site_name}/m2_coords_{site_name}.mat'
loaded_data = loadmat(m2_mat_filename)
#dzs_soil  = loaded_data['dzs_soil'].flatten()
#dzs_geo   = loaded_data['dzs_geo'].flatten()
#m2_coords = loaded_data['m2_coords']
loaded_gdf_dict = loaded_data['gdf_data']
gdf_reloaded = pd.DataFrame({
    'lon': loaded_gdf_dict['lon'][0, 0].flatten(),
    'lat': loaded_gdf_dict['lat'][0, 0].flatten(),
    'h_distance': loaded_gdf_dict['h_distance'][0, 0].flatten(),
    'elevation': loaded_gdf_dict['elevation'][0, 0].flatten()
})
geometry = [Point(xy) for xy in zip(gdf_reloaded['lon'], gdf_reloaded['lat'])]
hillslope_gdf = gpd.GeoDataFrame(gdf_reloaded, geometry=geometry)

# create hillslope polygon and shape object
xsec_plg = Polygon([hillslope_gdf.geometry[i] for i in range(hillslope_gdf.shape[0])])
xsec_plg_dict = {"type": "Feature", "id":0, "properties":{}, "geometry": mapping(xsec_plg)}
xsec_plg_dict_shply = watershed_workflow.utils.create_shply(xsec_plg_dict)

proj_daymet = "+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +datum=WGS84" # daymet crs
proj_wgs84  = "epsg:4326" # latlon
crs_daymet  = watershed_workflow.crs.from_string(proj_daymet)
crs_wgs84   = watershed_workflow.crs.from_string(proj_wgs84)

# convert to destination crs crs_wgs84
reproj_bnd = watershed_workflow.warp.shape(xsec_plg_dict, crs_daymet, crs_wgs84)
reproj_bnd_shply = watershed_workflow.utils.create_shply(reproj_bnd)

In [ ]:
gdf_reloaded

In [ ]:
def read_daymet_h5(filename):
    data = {}
    with h5py.File(filename, 'r') as f:
        for k, v in f.items():
            try:
                data[k] = v[:]
            except TypeError:
                data_t = {}
                for tk, tv in v.items():
                    data_t[tk] = tv[:]
                data[k] = data_t
    return data

In [ ]:
def write_daymet_h5(filename, data):
    with h5py.File(filename, 'w') as f:
        for k, v in data.items():
            try:
                f.create_dataset(k, data=v)
            except TypeError:
                g = f.create_group(k)
                for tk, tv in v.items():
                    g.create_dataset(tk, data=tv)

In [ ]:
def check_keys(data):
    keys = data.keys()
    for key in keys:
        print(key, type(data[key]))
        if isinstance(data[key], dict):
            _keys = data[key].keys()
            for _key in _keys:
                if int(_key) > 5:
                    break
                print('\t', _key, ': type is', type(data[key][_key]))

In [ ]:
# print("mpl - figure.facecolor:", mpl.rcParams["figure.facecolor"])
# print("mpl - axes.facecolor:", mpl.rcParams["axes.facecolor"])
# print("mpl - savefig.facecolor:", mpl.rcParams["savefig.facecolor"]) # in the IPython notebook with an inline backend, where the "saved" version of the figure you see below the cell is not controlled by the figure parameter, but by the savefig paramter.
# print("plt - axes.facecolor:", plt.rcParams['axes.facecolor'])
# print("plt - grid.color:", plt.rcParams['grid.color'])
# print("Active style sheets:", plt.style.available)

# mpl.rcParams.update(mpl.rcParamsDefault)
# plt.style.use('default')
# mpl.rcParams["savefig.facecolor"] = "white"

# # Verify the reset worked:
# print("After reset - axes.facecolor:", plt.rcParams['axes.facecolor'])
# print("After reset - grid.color:", plt.rcParams['grid.color'])

In [ ]:
#plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 100
# plt.rcParams['figure.figsize'] = (10,4)
plt.rcParams['lines.linewidth'] = 1
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

# Load ELM results

In [ ]:
# Use the first spinup source file only as a geometry/variable probe.
# Phase extraction below opens the configured ELM realization for each requested date.
probe_run = config['spinup']['elm_run']
probe_date = spinup_dates[0]
path = f"{config['elm_root']}/ELM_MOSART_CONUS.{probe_run}/run/"
years = np.array([probe_date.year])


In [ ]:
f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{years[-1]}-01-01-00000.nc'
# f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.2021-08-05-00000.nc'
data = xr.open_dataset(f)
data

## Get keys and units

In [ ]:
keys, names, units = [], [], []
for key in list(data.keys()):
    try:
        keys.append(key)
    except:
        keys.append('')
    try:
        names.append(data[key].long_name)
    except:
        names.append('')
    try:
        units.append(data[key].units)
    except:
        units.append('')
df = pd.DataFrame(data={'keys': keys, 'names': names, 'units': units})
# df.to_csv('all_ELM_vars.csv')
df

In [ ]:
# search keyword
search_keyword = 'NH4'
idx = []
for i in range(len(df)):
    if search_keyword in df['keys'][i] or search_keyword in df['names'][i] or search_keyword in df['units'][i]:
        idx.append(i)
df.iloc[idx, :]

In [ ]:
# search keyword
search_keyword = 'NO3'
idx = []
for i in range(len(df)):
    if search_keyword in df['keys'][i] or search_keyword in df['names'][i] or search_keyword in df['units'][i]:
        idx.append(i)
df.iloc[idx, :]

In [ ]:
# search keyword
search_keyword = '_vr'
idx = []
for i in range(len(df)):
    if search_keyword in df['keys'][i] or search_keyword in df['names'][i] or search_keyword in df['units'][i]:
        idx.append(i)
df.iloc[idx, :]

In [ ]:
# search keyword
search_keyword = 'infiltration'
idx = []
for i in range(len(df)):
    if search_keyword in df['keys'][i] or search_keyword in df['names'][i] or search_keyword in df['units'][i]:
        idx.append(i)
df.iloc[idx, :]

## Plot ELM grid, watersheds boundary, and hillslope
- plot soil moisture - top layer - with Naches/Oak Creek/hillslope


In [ ]:
lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
X, Y = np.meshgrid(lon, lat)

# Flatten X, Y into points
elm_mesh_points = np.column_stack([X.ravel(), Y.ravel()])
tree = cKDTree(elm_mesh_points)
line_coords = list(reproj_bnd_shply.exterior.coords)

# Find nearest cell for each point along the line
distances, indices_flat = tree.query(line_coords)
indices_flat = np.unique(indices_flat)

# Convert flat indices back to 2D indices
hillslope_elm_lat_indices = indices_flat // X.shape[1]
hillslope_elm_lon_indices = indices_flat % X.shape[1]

print(X.shape)
print([len(lat), len(lon)])
print(hillslope_elm_lat_indices)
print(hillslope_elm_lon_indices)

In [ ]:
fig, [ax1, ax2, ax3] = plt.subplots(1, 3, figsize=(12,4), subplot_kw={'projection': cartopy.crs.LambertConformal()})
i=0 # 1st layer

# raw
EasyMap("10m", ax=ax1, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
ax1.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
ax1.add_feature(cartopy.feature.RIVERS)
art = ax1.pcolormesh(X, Y, data['H2OSOI'].values[60, i, :, :], cmap='Spectral_r', vmin=0, vmax=1, transform=cartopy.crs.PlateCarree())
fig.colorbar(art, label='[mm3/mm3]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
ax1.set_title(f'layer {i+1}: {str(np.around(data.levgrnd.values[i], 3))} m')
# add specified boundaries
ax1.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k--', lw=2, transform=pc)
ax1.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
watershed_workflow.plot.shply(reproj_bnd_shply, crs_wgs84, ax=ax1, color='r')

# zoom in level1
lon_min_zoom = np.min(oakcreek_wbd[:, 0])+360
lon_max_zoom = np.max(oakcreek_wbd[:, 0])+360
lat_min_zoom = np.min(oakcreek_wbd[:, 1])
lat_max_zoom = np.max(oakcreek_wbd[:, 1])

EasyMap("10m", ax=ax2, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
ax2.add_feature(cartopy.feature.RIVERS)
art = ax2.pcolormesh(X, Y, data['H2OSOI'].values[60, i, :, :], cmap='Spectral_r', vmin=0, vmax=1, transform=cartopy.crs.PlateCarree())
fig.colorbar(art, label='[mm3/mm3]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
ax2.set_title(f'layer {i+1}: {str(np.around(data.levgrnd.values[i], 3))} m')
# add specified boundaries
#ax2.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k--', lw=2, transform=pc)
ax2.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
watershed_workflow.plot.shply(reproj_bnd_shply, crs_wgs84, ax=ax2, color='r')
ax2.set_extent([lon_min_zoom, lon_max_zoom, lat_min_zoom, lat_max_zoom], crs=cartopy.crs.PlateCarree())

# zoom in level2
lon_min_zoom, lat_min_zoom, lon_max_zoom, lat_max_zoom = reproj_bnd_shply.bounds
lon_min_zoom +=360
lon_max_zoom +=360
buffer = 0.005

#EasyMap("10m", ax=ax3, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
#ax3.add_feature(cartopy.feature.RIVERS)
art = ax3.pcolormesh(X, Y, data['H2OSOI'].values[60, i, :, :], cmap='Spectral_r', vmin=0, vmax=1, transform=cartopy.crs.PlateCarree())
fig.colorbar(art, label='[mm3/mm3]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
ax3.set_title(f'layer {i+1}: {str(np.around(data.levgrnd.values[i], 3))} m')
# add specified boundaries
#ax3.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k--', lw=2, transform=pc)
ax3.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
watershed_workflow.plot.shply(reproj_bnd_shply, crs_wgs84, ax=ax3, color='r')
if False: # can make it True after found overlap ELM grids
    ax3.scatter(X[hillslope_elm_lat_indices, hillslope_elm_lon_indices], 
                Y[hillslope_elm_lat_indices, hillslope_elm_lon_indices], 
               s=50, c='blue', marker='x', linewidths=2,
               transform=cartopy.crs.PlateCarree(), 
               label='Nearest cells', zorder=10)
ax3.set_extent([lon_min_zoom - buffer, lon_max_zoom + buffer, lat_min_zoom - buffer, lat_max_zoom + buffer], crs=cartopy.crs.PlateCarree())

plt.show()

# Get ELM for the hillslope site

similar to the `get_Daymet.ipynb`
- first, interpolate data to x y coordinate of 2D hillslope -> `raw_dat_2dtran`
- then, warp `raw_dat_2dtran` to coordinate (0,680)x(0,1) -> `raw_dat_2dtran_warped`
- lastly, write to hdf5 file

In [ ]:
# The source window is a sequence of complete no-leap water years.  Do not infer
# its five blocks from calendar labels: 2012-10-01 through 2017-09-30 spans six
# calendar years but exactly five 365-day source years.
if len(spinup_dates) % 365 != 0:
    raise ValueError('spinup source period must contain complete 365-day water years')
spinup_source_years = len(spinup_dates) // 365
print(f"Spinup source: {spinup_label} ({len(spinup_dates)} days; {spinup_source_years} water years)")
print(f"Prefire:       {prefire_label} ({len(prefire_dates)} days)")
print(f"Postfire:      {postfire_label} ({len(postfire_dates)} days)" if postfire_dates else 'Postfire:      not configured')


## Find value at x and y in 2D transect

Use Zhi's strategy in `read_ELM_byZhi.ipynb`

In [ ]:
# 1. read ELM Naches lon and lat
lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
X, Y = np.meshgrid(lon, lat)

# 2. convert lon lot to crs_daymet
lonlat = np.zeros((len(X.flatten()), 2))
lonlat[:, 0], lonlat[:, 1] = X.flatten(),  Y.flatten()
#proj_lcc = "+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +datum=WGS84" # daymet crs
#proj_wgs84 = "epsg:4326" # latlon
lonlat_to_daymet = np.array(Transformer.from_crs(proj_wgs84, proj_daymet).transform(lonlat[:, 1], lonlat[:, 0]))

In [ ]:
# 3. 2d hillslope mesh in daymet crs
fig, axes = plt.subplots(1, 2, figsize=(12,4))
ax = axes.flatten()

ax[0].scatter(lonlat_to_daymet[0], lonlat_to_daymet[1], s=0.5, c='red')
ax[0].scatter(gdf_reloaded['lon'], gdf_reloaded['lat'], s=0.5, c='blue')
ax[0].axis('equal')

ax[1].scatter(lonlat_to_daymet[0], lonlat_to_daymet[1], s=2, c='red')
ax[1].scatter(gdf_reloaded['lon'], gdf_reloaded['lat'], s=2, c='blue')
# Set axis limits based on gdf_reloaded extent with a small buffer
lon_buffer = (gdf_reloaded['lon'].max() - gdf_reloaded['lon'].min()) * 3
lat_buffer = (gdf_reloaded['lat'].max() - gdf_reloaded['lat'].min()) * 10
ax[1].set_xlim(gdf_reloaded['lon'].min() - lon_buffer, 
               gdf_reloaded['lon'].max() + lon_buffer)
ax[1].set_ylim(gdf_reloaded['lat'].min() - lat_buffer, 
               gdf_reloaded['lat'].max() + lat_buffer)
ax[1].set_aspect('equal', adjustable='box')  # Use set_aspect instead of axis('equal')

plt.tight_layout()
plt.show()

In [ ]:
# 4. prepare interpolation
xy_source = np.zeros((len(lonlat_to_daymet[0]), 2))
xy_source[:, 0], xy_source[:, 1] = lonlat_to_daymet[0], lonlat_to_daymet[1]

target_x_h5file = gdf_reloaded['h_distance']
target_y_h5file = np.array([0.0, 1.0])

xy_target = np.zeros((len(gdf_reloaded['lon']) * 2, 2))
xy_target[:, 0] = np.concatenate([gdf_reloaded['lon'], gdf_reloaded['lon']])
xy_target[:, 1] = np.concatenate([gdf_reloaded['lat'], gdf_reloaded['lat']])

In [ ]:
#target_x_h5file # 0-680
#xy_source[:, 0] # easting
#xy_source[:, 1] # northing
#xy_target[:, 0] # easting
#xy_target[:, 1] # northing

In [ ]:
def idw_interpolation(xy_target, xy_source, values, power=2):
    tree = cKDTree(xy_source)
    distances, indices = tree.query(xy_target, k=4)
    weights = 1 / (distances ** power)
    weights /= weights.sum(axis=1, keepdims=True)
    interpolated_values = np.sum(values[indices] * weights, axis=1)
    return interpolated_values

In [ ]:
# 5. Phase-aware ELM total-DOC and CN extraction.
# This notebook intentionally writes only total DOC. Lambda donor-bin conversion
# belongs in 2-add_reaction_lambda.ipynb.
f_DOM, fdom = 0.01, '001'
k_in_sec = np.array([0.1, 1.5]) / 3600.0
k, k_label = k_in_sec[1], '15'

def elm_history_sample(elm_run, current_date):
    """Resolve annual ELM output or one-file-per-day ELM output."""
    run_dir = Path(config['elm_root']) / f'ELM_MOSART_CONUS.{elm_run}' / 'run'
    prefix = f'ELM_MOSART_CONUS.{elm_run}.elm.h0.'
    # A daily run's 1 January filename is indistinguishable from an annual file.
    # Its adjacent 2 January file identifies the daily layout before indexing.
    daily = run_dir / f'{prefix}{current_date.isoformat()}-00000.nc'
    daily_layout_marker = run_dir / f'{prefix}{current_date.year}-01-02-00000.nc'
    if daily_layout_marker.exists() and daily.exists():
        return daily, 0
    annual = run_dir / f'{prefix}{current_date.year}-01-01-00000.nc'
    if annual.exists():
        return annual, noleap_day_of_year(current_date)
    if daily.exists():
        return daily, 0
    raise FileNotFoundError(f'missing ELM sample for {current_date}: expected {annual} or {daily}')

def empty_phase_data(ndays):
    return ({'DOC production [molC m^-3 s^-1]': {}, 'time [s]': np.arange(ndays) * 86400,
             'x [m]': target_x_h5file, 'y [m]': target_y_h5file},
            {'NH4+ bulk volume basis [molS L^-1]': np.zeros(ndays),
             'NO3- bulk volume basis [molS L^-1]': np.zeros(ndays),
             'NH4+ mol water basis [molS molH^-1]': np.zeros(ndays),
             'NO3- mol water basis [molS molH^-1]': np.zeros(ndays),
             'DOC bulk volume basis v1 [molS L^-1]': np.zeros(ndays),
             'DOC mol water basis v1 [molS molH^-1]': np.zeros(ndays),
             'time [s]': np.arange(ndays) * 86400})

def extract_total_doc_cn_phase(phase_name, dates, elm_run, day_run_overrides=None):
    """Extract one exact no-leap phase using its configured ELM realization."""
    flux, conc = empty_phase_data(len(dates))
    day_run_overrides = day_run_overrides or {}
    data, open_filename = None, None
    for day_index, current_date in enumerate(tqdm(dates, desc=phase_name)):
        current_run = day_run_overrides.get(current_date, elm_run)
        filename, day = elm_history_sample(current_run, current_date)
        if filename != open_filename:
            if data is not None:
                data.close()
            data, open_filename = xr.open_dataset(filename), filename
        label = str(day_index)
        hr = data['HR'].values[day, :, :] * f_DOM / soil_thickness_median / 12
        flux['DOC production [molC m^-3 s^-1]'][label] = idw_interpolation(
            xy_target, xy_source, hr.ravel()).reshape(len(target_y_h5file), len(target_x_h5file))
        water = data['H2OSOI'].values[day, :, hillslope_elm_lat_indices, hillslope_elm_lon_indices]
        water_layers = water[:, :num_soil_layer_thickness_median].mean(axis=0)
        water_bulk = np.sum(water_layers * soil_layer_thickness_array) / np.sum(soil_layer_thickness_array)
        if water_bulk <= 0:
            raise ValueError(f'{phase_name}: non-positive hillslope water volume on {current_date}')
        for elm_key, bulk_key, water_key in [
            ('SMIN_NH4_vr', 'NH4+ bulk volume basis [molS L^-1]', 'NH4+ mol water basis [molS molH^-1]'),
            ('SMIN_NO3_vr', 'NO3- bulk volume basis [molS L^-1]', 'NO3- mol water basis [molS molH^-1]')]:
            values = data[elm_key].values[day, :, hillslope_elm_lat_indices, hillslope_elm_lon_indices] / 14 / 1000
            values_soil = values[:, :num_soil_layer_thickness_median].mean(axis=0)
            bulk = np.sum(values_soil * soil_layer_thickness_array) / np.sum(soil_layer_thickness_array)
            conc[bulk_key][day_index] = bulk
            conc[water_key][day_index] = bulk / water_bulk * 1000 / rho_m
        doc_bulk = flux['DOC production [molC m^-3 s^-1]'][label].mean() / 1000
        conc['DOC bulk volume basis v1 [molS L^-1]'][day_index] = doc_bulk
        conc['DOC mol water basis v1 [molS molH^-1]'][day_index] = doc_bulk / water_bulk * 1000 / rho_m
    if data is not None:
        data.close()
    return flux, conc

def write_total_doc(path, flux):
    write_daymet_h5(path, flux)

def write_total_cn(path, conc):
    with h5.File(path, 'w') as hdf:
        hdf.create_dataset('Time', data=conc['time [s]'])
        for key, values in conc.items():
            if key != 'time [s]':
                hdf.create_dataset(key, data=values / k if key.startswith('DOC') else values)

def concatenate_flux(phases):
    arrays = [phase['DOC production [molC m^-3 s^-1]'][str(i)]
              for phase in phases for i in range(len(phase['time [s]']))]
    return {'DOC production [molC m^-3 s^-1]': {str(i): value for i, value in enumerate(arrays)},
            'time [s]': np.arange(len(arrays)) * 86400,
            'x [m]': phases[0]['x [m]'], 'y [m]': phases[0]['y [m]']}

def concatenate_cn(phases):
    keys = [key for key in phases[0] if key != 'time [s]']
    merged = {key: np.concatenate([phase[key] for phase in phases]) for key in keys}
    merged['time [s]'] = np.arange(len(merged[keys[0]])) * 86400
    return merged


### Find soil layers in ELM 
- "soil" layers info. is used to obtain the averaged [NH4+] and [NO3-]
- "soil" layers info. is also used to obtain the overall water content, to convert concentrations per bulk volume to concentrations per water volume or per mole water
- based on soil_thickness_median from 1-main_workflow_OakCreek.NF01.ats1.5.ipynb

In [ ]:
for layer in range(len(data.coords["levgrnd"])):
    tmp_total_thickness = sum(data.coords["levgrnd"][0:layer+1]).item()
    print(tmp_total_thickness)
    if tmp_total_thickness > soil_thickness_median:
        num_soil_layer_thickness_median = layer + 1
        break
soil_layer_thickness_array = (data.coords["levgrnd"][0:num_soil_layer_thickness_median]).values

print(num_soil_layer_thickness_median) # number of layers used to calculate the NH4+ and NO3+
print(soil_layer_thickness_array)
print(sum(soil_layer_thickness_array))

### Find values for DOC injection and transport BC

In [ ]:
# Extract each configured ELM phase with its own realization.
elm_flux_data_spinup_raw, elm_conc_data_spinup_raw = extract_total_doc_cn_phase(
    'spinup source', spinup_dates, config['spinup']['elm_run'])
elm_flux_data_prefire, elm_conc_data_prefire = extract_total_doc_cn_phase(
    'prefire', prefire_dates, config['prefire_transient']['elm_run'])
if postfire_dates:
    ignition_run = config['postfire_transient'].get('ignition_day_elm_run')
    ignition_override = {postfire_dates[0]: ignition_run} if ignition_run else {}
    elm_flux_data_postfire, elm_conc_data_postfire = extract_total_doc_cn_phase(
        'postfire', postfire_dates, config['postfire_transient']['elm_run'], ignition_override)
else:
    elm_flux_data_postfire = elm_conc_data_postfire = None
elm_flux_data_transient, elm_conc_data_transient = elm_flux_data_prefire, elm_conc_data_prefire


In [ ]:
# Postfire is handled by the same phase extractor above when configured.


In [ ]:
# Spinup source extraction is handled by the same phase extractor above.


In [ ]:
soil_layer_thickness_array/sum(soil_layer_thickness_array)

## write hdf5 files

In [ ]:
# DOC concentration conversion remains k = 1.5 h^-1 (stored as `k` above).


### for case cybernetic transient run

In [ ]:
# Write exact-date total forcing products for the transient phases.
write_total_doc(forcing_prefire_dir / f'{site_name}_DOC_source.h5', elm_flux_data_prefire)
write_total_cn(forcing_prefire_dir / f'{site_name}_CNbc_conc.h5', elm_conc_data_prefire)
if postfire_dates:
    write_total_doc(forcing_postfire_dir / f'{site_name}_DOC_source.h5', elm_flux_data_postfire)
    write_total_cn(forcing_postfire_dir / f'{site_name}_CNbc_conc.h5', elm_conc_data_postfire)
print(f'Wrote prefire total DOC/CN: {forcing_prefire_dir}')
if postfire_dates:
    print(f'Wrote postfire total DOC/CN: {forcing_postfire_dir}')


### for case cyclic spinup
1. caseflow steady state spinup run0
2. caseflow cyclic spinup run1
3. **casecybernetic cyclic spinup run1**
4. ~~casecybernetic transient~~

In [ ]:
# Average sequential source water years into a 365-day typical year and tile it.
source_flux_arrays = [elm_flux_data_spinup_raw['DOC production [molC m^-3 s^-1]'][str(i)] for i in range(len(spinup_dates))]
typical_flux = np.mean(np.stack(source_flux_arrays).reshape(spinup_source_years, 365, *source_flux_arrays[0].shape), axis=0)
elm_flux_data_spinup = {'DOC production [molC m^-3 s^-1]': {str(i): typical_flux[i % 365] for i in range(nyears_cyclic_spinup * 365)},
                         'time [s]': np.arange(nyears_cyclic_spinup * 365) * 86400,
                         'x [m]': target_x_h5file, 'y [m]': target_y_h5file}
elm_conc_data_spinup = {'time [s]': np.arange(nyears_cyclic_spinup * 365) * 86400}
for key, values in elm_conc_data_spinup_raw.items():
    if key != 'time [s]':
        elm_conc_data_spinup[key] = np.tile(values.reshape(spinup_source_years, 365).mean(axis=0), nyears_cyclic_spinup)
spinup_doc_file = forcing_spinup_dir / f'{site_name}_DOC_source_cyclic{nyears_cyclic_spinup}y.h5'
spinup_cn_file = forcing_spinup_dir / f'{site_name}_CNbc_conc_cyclic{nyears_cyclic_spinup}y.h5'
write_total_doc(spinup_doc_file, elm_flux_data_spinup)
write_total_cn(spinup_cn_file, elm_conc_data_spinup)
phases_flux, phases_cn = [elm_flux_data_spinup, elm_flux_data_prefire], [elm_conc_data_spinup, elm_conc_data_prefire]
if postfire_dates:
    phases_flux.append(elm_flux_data_postfire)
    phases_cn.append(elm_conc_data_postfire)
full_flux, full_cn = concatenate_flux(phases_flux), concatenate_cn(phases_cn)
expected_days = nyears_cyclic_spinup * 365 + len(prefire_dates) + len(postfire_dates)
for name, times in [('DOC source', full_flux['time [s]']), ('CN boundary', full_cn['time [s]'])]:
    if len(times) != expected_days or not np.array_equal(times, np.arange(expected_days) * 86400):
        raise ValueError(f'{name} merge is not a continuous daily forcing timeline')
write_total_doc(forcing_full_dir / f'{site_name}_DOC_source.h5', full_flux)
write_total_cn(forcing_full_dir / f'{site_name}_CNbc_conc.h5', full_cn)
print('ELM total-DOC/CN forcing summary')
print(f'  spinup source: {spinup_label}, {len(spinup_dates)} days')
print(f'  cyclic spinup: {nyears_cyclic_spinup} years, {len(elm_flux_data_spinup["time [s]"])} days')
print(f'  prefire:       {prefire_label}, {len(prefire_dates)} days')
print(f'  postfire:      {postfire_label + ", " + str(len(postfire_dates)) + " days" if postfire_dates else "not configured"}')
print(f'  total:         {expected_days} samples; {full_flux["time [s]"][0]} to {full_flux["time [s]"][-1]} s')
print(f'Wrote canonical total DOC/CN: {forcing_full_dir}')


In [ ]:
# Replaced by the phase-aware extraction and merge cells above.


In [ ]:
# Replaced by the phase-aware extraction and merge cells above.


In [ ]:
# Replaced by the phase-aware extraction and merge cells above.


## plot domain average over time

### plot DOC injection flux

In [ ]:
# Domain-mean total DOC injection for the three model-forcing phases.
# Each panel uses its phase-local clock; titles preserve forcing provenance.
phase_fluxes = [
    ('Cyclic spinup', f'{nyears_cyclic_spinup} y cyclic; source {spinup_label}', elm_flux_data_spinup),
    ('Prefire transient', prefire_label, elm_flux_data_prefire),
]
if postfire_dates:
    phase_fluxes.append(('Postfire transient', postfire_label, elm_flux_data_postfire))

fig, axes = plt.subplots(1, len(phase_fluxes), figsize=(6 * len(phase_fluxes), 4), squeeze=False)
for ax, (name, period, flux) in zip(axes.flat, phase_fluxes):
    values = np.array([flux['DOC production [molC m^-3 s^-1]'][str(i)].mean()
                       for i in range(len(flux['time [s]']))])
    ax.plot(flux['time [s]'] / 86400, values, color='tab:green', linewidth=0.7)
    ax.set_title(f'{name}\n{period}')
    ax.set_xlabel('Phase-local simulation day')
    ax.set_ylabel('Mean DOC production [molC m$^{-3}$ s$^{-1}$]')
    ax.grid(alpha=0.3)
fig.suptitle('ELM total DOC injection forcing', y=1.04)
plt.tight_layout()
plt.show()


### plot concentrations

In [ ]:
# Generated CN boundary conditions for the same three forcing phases.
# DOC is divided by k here because that is the value written to the CN HDF5 file.
phase_concentrations = [
    ('Cyclic spinup', f'{nyears_cyclic_spinup} y cyclic; source {spinup_label}', elm_conc_data_spinup),
    ('Prefire transient', prefire_label, elm_conc_data_prefire),
]
if postfire_dates:
    phase_concentrations.append(('Postfire transient', postfire_label, elm_conc_data_postfire))

fig, axes = plt.subplots(1, len(phase_concentrations), figsize=(6 * len(phase_concentrations), 4), squeeze=False)
for ax, (name, period, conc) in zip(axes.flat, phase_concentrations):
    day = conc['time [s]'] / 86400
    ax.semilogy(day, np.maximum(conc['NH4+ mol water basis [molS molH^-1]'], np.finfo(float).tiny), label='NH4+', linewidth=0.7)
    ax.semilogy(day, np.maximum(conc['NO3- mol water basis [molS molH^-1]'], np.finfo(float).tiny), label='NO3-', linewidth=0.7)
    ax.semilogy(day, np.maximum(conc['DOC mol water basis v1 [molS molH^-1]'] / k, np.finfo(float).tiny), label='Total DOC', linewidth=0.7)
    ax.set_title(f'{name}\n{period}')
    ax.set_xlabel('Phase-local simulation day')
    ax.set_ylabel('Boundary concentration [molS molH$^{-1}$]')
    ax.grid(alpha=0.3, which='both')
    ax.legend()
fig.suptitle('ELM CN boundary forcing written to HDF5', y=1.04)
plt.tight_layout()
plt.show()


# [Explore] Carbon pool concentrations

## plot LIT and SOM pool concentrations
- [DOC] v1, estimated based on quasi-steady-state assumption
- [DOC] v2, estimated based on P_DOC/(P-ET)

In [ ]:
# Replaced by the phase-aware extraction and merge cells above.


In [ ]:
# Replaced by the phase-aware extraction and merge cells above.
